
# Hospital EDW — Unity Catalog PII & PHI Governance Tagging

This notebook sets table-level and column-level tags across the **Silver** and **Gold** layers in `hospital_catalog`.

### Governance Standards Applied:
- **HIPAA Safe Harbor**: Tagging 18 identifiers (names, dates, geographic data, phone numbers, MRNs) as `phi = 'true'`.
- **GDPR / ISO 27701**: Tagging direct & quasi-identifiers as `pii = 'true'`.
- **Data Classifications**: `restricted` (patient clinical/PII), `confidential` (billing/financial), `internal` (doctors/staff), `public` (reference dimensions).


## Part 1: Silver Layer — EMR Domain (Patients & Doctors)

In [ ]:
%sql
-- 1.1 silver.patients
ALTER TABLE hospital_catalog.silver.patients
SET TAGS (
  'contains_pii' = 'true',
  'contains_phi' = 'true',
  'classification' = 'restricted',
  'domain' = 'emr',
  'retention_policy' = '7_years'
);

ALTER TABLE hospital_catalog.silver.patients ALTER COLUMN patient_id
SET TAGS ('pii' = 'true', 'data_type' = 'pseudonymized_key', 'info_type' = 'patient_mrn');

ALTER TABLE hospital_catalog.silver.patients ALTER COLUMN first_name
SET TAGS ('pii' = 'true', 'data_type' = 'direct_identifier', 'info_type' = 'first_name');

ALTER TABLE hospital_catalog.silver.patients ALTER COLUMN last_name
SET TAGS ('pii' = 'true', 'data_type' = 'direct_identifier', 'info_type' = 'last_name');

ALTER TABLE hospital_catalog.silver.patients ALTER COLUMN gender
SET TAGS ('pii' = 'false', 'data_type' = 'quasi_identifier', 'info_type' = 'demographic_gender');

ALTER TABLE hospital_catalog.silver.patients ALTER COLUMN date_of_birth
SET TAGS ('pii' = 'true', 'phi' = 'true', 'data_type' = 'quasi_identifier', 'info_type' = 'date_of_birth');

ALTER TABLE hospital_catalog.silver.patients ALTER COLUMN contact_no
SET TAGS ('pii' = 'true', 'data_type' = 'direct_identifier', 'info_type' = 'phone_number');

ALTER TABLE hospital_catalog.silver.patients ALTER COLUMN address
SET TAGS ('pii' = 'true', 'data_type' = 'direct_identifier', 'info_type' = 'physical_address');

In [ ]:
%sql
-- 1.2 silver.doctors
ALTER TABLE hospital_catalog.silver.doctors
SET TAGS (
  'contains_pii' = 'true',
  'classification' = 'internal',
  'domain' = 'emr'
);

ALTER TABLE hospital_catalog.silver.doctors ALTER COLUMN doctor_id
SET TAGS ('pii' = 'true', 'data_type' = 'pseudonymized_key', 'info_type' = 'staff_id');

ALTER TABLE hospital_catalog.silver.doctors ALTER COLUMN first_name
SET TAGS ('pii' = 'true', 'data_type' = 'direct_identifier', 'info_type' = 'staff_name');

ALTER TABLE hospital_catalog.silver.doctors ALTER COLUMN last_name
SET TAGS ('pii' = 'true', 'data_type' = 'direct_identifier', 'info_type' = 'staff_name');

ALTER TABLE hospital_catalog.silver.doctors ALTER COLUMN contact_no
SET TAGS ('pii' = 'true', 'data_type' = 'direct_identifier', 'info_type' = 'business_phone_number');


## Part 2: Silver Layer — Clinical Events (Encounters, Diagnoses, Procedures, Admissions)

In [ ]:
%sql
-- 2.1 silver.encounters
ALTER TABLE hospital_catalog.silver.encounters
SET TAGS (
  'contains_phi' = 'true',
  'classification' = 'restricted',
  'domain' = 'emr'
);

ALTER TABLE hospital_catalog.silver.encounters ALTER COLUMN patient_id
SET TAGS ('pii' = 'true', 'data_type' = 'pseudonymized_key', 'info_type' = 'patient_mrn');

ALTER TABLE hospital_catalog.silver.encounters ALTER COLUMN doctor_id
SET TAGS ('pii' = 'true', 'data_type' = 'pseudonymized_key', 'info_type' = 'staff_id');

ALTER TABLE hospital_catalog.silver.encounters ALTER COLUMN encounter_date
SET TAGS ('phi' = 'true', 'data_type' = 'quasi_identifier', 'info_type' = 'encounter_timestamp');

ALTER TABLE hospital_catalog.silver.encounters ALTER COLUMN reason
SET TAGS ('phi' = 'true', 'data_type' = 'clinical_observation', 'info_type' = 'chief_complaint');

In [ ]:
%sql
-- 2.2 silver.diagnoses
ALTER TABLE hospital_catalog.silver.diagnoses
SET TAGS (
  'contains_phi' = 'true',
  'classification' = 'restricted',
  'domain' = 'emr'
);

ALTER TABLE hospital_catalog.silver.diagnoses ALTER COLUMN patient_id
SET TAGS ('pii' = 'true', 'data_type' = 'pseudonymized_key', 'info_type' = 'patient_mrn');

ALTER TABLE hospital_catalog.silver.diagnoses ALTER COLUMN doctor_id
SET TAGS ('pii' = 'true', 'data_type' = 'pseudonymized_key', 'info_type' = 'staff_id');

ALTER TABLE hospital_catalog.silver.diagnoses ALTER COLUMN diagnosis_name
SET TAGS ('phi' = 'true', 'data_type' = 'clinical_observation', 'info_type' = 'medical_condition');

ALTER TABLE hospital_catalog.silver.diagnoses ALTER COLUMN icd10_code
SET TAGS ('phi' = 'true', 'data_type' = 'clinical_observation', 'info_type' = 'icd10_code');

ALTER TABLE hospital_catalog.silver.diagnoses ALTER COLUMN diagnosis_date
SET TAGS ('phi' = 'true', 'data_type' = 'quasi_identifier', 'info_type' = 'clinical_date');

In [ ]:
%sql
-- 2.3 silver.procedures
ALTER TABLE hospital_catalog.silver.procedures
SET TAGS (
  'contains_phi' = 'true',
  'classification' = 'restricted',
  'domain' = 'emr'
);

ALTER TABLE hospital_catalog.silver.procedures ALTER COLUMN patient_id
SET TAGS ('pii' = 'true', 'data_type' = 'pseudonymized_key', 'info_type' = 'patient_mrn');

ALTER TABLE hospital_catalog.silver.procedures ALTER COLUMN surgeon_id
SET TAGS ('pii' = 'true', 'data_type' = 'pseudonymized_key', 'info_type' = 'staff_id');

ALTER TABLE hospital_catalog.silver.procedures ALTER COLUMN procedure_name
SET TAGS ('phi' = 'true', 'data_type' = 'clinical_observation', 'info_type' = 'surgical_procedure');

ALTER TABLE hospital_catalog.silver.procedures ALTER COLUMN procedure_date
SET TAGS ('phi' = 'true', 'data_type' = 'quasi_identifier', 'info_type' = 'clinical_date');

ALTER TABLE hospital_catalog.silver.procedures ALTER COLUMN notes
SET TAGS ('phi' = 'true', 'data_type' = 'clinical_observation', 'info_type' = 'operative_notes');

In [ ]:
%sql
-- 2.4 silver.admissions
ALTER TABLE hospital_catalog.silver.admissions
SET TAGS (
  'contains_phi' = 'true',
  'classification' = 'restricted',
  'domain' = 'emr'
);

ALTER TABLE hospital_catalog.silver.admissions ALTER COLUMN patient_id
SET TAGS ('pii' = 'true', 'data_type' = 'pseudonymized_key', 'info_type' = 'patient_mrn');

ALTER TABLE hospital_catalog.silver.admissions ALTER COLUMN admission_date
SET TAGS ('phi' = 'true', 'data_type' = 'quasi_identifier', 'info_type' = 'admission_date');

ALTER TABLE hospital_catalog.silver.admissions ALTER COLUMN discharge_date
SET TAGS ('phi' = 'true', 'data_type' = 'quasi_identifier', 'info_type' = 'discharge_date');

ALTER TABLE hospital_catalog.silver.admissions ALTER COLUMN ward_name
SET TAGS ('phi' = 'true', 'data_type' = 'location_tracking', 'info_type' = 'hospital_ward');

ALTER TABLE hospital_catalog.silver.admissions ALTER COLUMN room_no
SET TAGS ('phi' = 'true', 'data_type' = 'location_tracking', 'info_type' = 'room_number');

ALTER TABLE hospital_catalog.silver.admissions ALTER COLUMN bed_no
SET TAGS ('phi' = 'true', 'data_type' = 'location_tracking', 'info_type' = 'bed_number');

ALTER TABLE hospital_catalog.silver.admissions ALTER COLUMN discharge_disposition
SET TAGS ('phi' = 'true', 'data_type' = 'clinical_observation', 'info_type' = 'clinical_outcome');


## Part 3: Silver Layer — Laboratory Domain

In [ ]:
%sql
-- 3.1 silver.lab_orders & silver.lab_results
ALTER TABLE hospital_catalog.silver.lab_orders
SET TAGS ('contains_phi' = 'true', 'classification' = 'restricted', 'domain' = 'laboratory');

ALTER TABLE hospital_catalog.silver.lab_orders ALTER COLUMN patient_id
SET TAGS ('pii' = 'true', 'data_type' = 'pseudonymized_key', 'info_type' = 'patient_mrn');

ALTER TABLE hospital_catalog.silver.lab_orders ALTER COLUMN doctor_id
SET TAGS ('pii' = 'true', 'data_type' = 'pseudonymized_key', 'info_type' = 'staff_id');

ALTER TABLE hospital_catalog.silver.lab_orders ALTER COLUMN test_name
SET TAGS ('phi' = 'true', 'data_type' = 'clinical_observation', 'info_type' = 'lab_test_name');

ALTER TABLE hospital_catalog.silver.lab_orders ALTER COLUMN order_date
SET TAGS ('phi' = 'true', 'data_type' = 'quasi_identifier', 'info_type' = 'clinical_date');


ALTER TABLE hospital_catalog.silver.lab_results
SET TAGS ('contains_phi' = 'true', 'classification' = 'restricted', 'domain' = 'laboratory');

ALTER TABLE hospital_catalog.silver.lab_results ALTER COLUMN patient_id
SET TAGS ('pii' = 'true', 'data_type' = 'pseudonymized_key', 'info_type' = 'patient_mrn');

ALTER TABLE hospital_catalog.silver.lab_results ALTER COLUMN test_name
SET TAGS ('phi' = 'true', 'data_type' = 'clinical_observation', 'info_type' = 'lab_test_name');

ALTER TABLE hospital_catalog.silver.lab_results ALTER COLUMN parameter_name
SET TAGS ('phi' = 'true', 'data_type' = 'clinical_observation', 'info_type' = 'test_parameter');

ALTER TABLE hospital_catalog.silver.lab_results ALTER COLUMN result_value
SET TAGS ('phi' = 'true', 'data_type' = 'clinical_observation', 'info_type' = 'lab_result_numeric');

ALTER TABLE hospital_catalog.silver.lab_results ALTER COLUMN reference_range
SET TAGS ('phi' = 'true', 'data_type' = 'clinical_observation', 'info_type' = 'reference_range');

ALTER TABLE hospital_catalog.silver.lab_results ALTER COLUMN abnormal_flag
SET TAGS ('phi' = 'true', 'data_type' = 'clinical_observation', 'info_type' = 'abnormal_flag');

ALTER TABLE hospital_catalog.silver.lab_results ALTER COLUMN result_date
SET TAGS ('phi' = 'true', 'data_type' = 'quasi_identifier', 'info_type' = 'clinical_date');


## Part 4: Silver Layer — Pharmacy & Billing Domains

In [ ]:
%sql
-- 4.1 Pharmacy
ALTER TABLE hospital_catalog.silver.prescriptions
SET TAGS ('contains_phi' = 'true', 'classification' = 'restricted', 'domain' = 'pharmacy');

ALTER TABLE hospital_catalog.silver.prescriptions ALTER COLUMN patient_id
SET TAGS ('pii' = 'true', 'data_type' = 'pseudonymized_key', 'info_type' = 'patient_mrn');

ALTER TABLE hospital_catalog.silver.prescriptions ALTER COLUMN doctor_id
SET TAGS ('pii' = 'true', 'data_type' = 'pseudonymized_key', 'info_type' = 'staff_id');

ALTER TABLE hospital_catalog.silver.prescriptions ALTER COLUMN medication_name
SET TAGS ('phi' = 'true', 'data_type' = 'clinical_observation', 'info_type' = 'prescription_drug');

ALTER TABLE hospital_catalog.silver.prescriptions ALTER COLUMN dosage
SET TAGS ('phi' = 'true', 'data_type' = 'clinical_observation', 'info_type' = 'medication_dosage');

ALTER TABLE hospital_catalog.silver.prescriptions ALTER COLUMN prescribed_date
SET TAGS ('phi' = 'true', 'data_type' = 'quasi_identifier', 'info_type' = 'clinical_date');


ALTER TABLE hospital_catalog.silver.pharmacy_transactions
SET TAGS ('contains_financial' = 'true', 'classification' = 'confidential', 'domain' = 'pharmacy');

ALTER TABLE hospital_catalog.silver.pharmacy_transactions ALTER COLUMN patient_id
SET TAGS ('pii' = 'true', 'data_type' = 'pseudonymized_key', 'info_type' = 'patient_mrn');

ALTER TABLE hospital_catalog.silver.pharmacy_transactions ALTER COLUMN total_amount
SET TAGS ('financial' = 'true', 'data_type' = 'monetary_amount', 'info_type' = 'charge_amount');

ALTER TABLE hospital_catalog.silver.pharmacy_transactions ALTER COLUMN payment_method
SET TAGS ('financial' = 'true', 'data_type' = 'monetary_amount', 'info_type' = 'payment_instrument');

In [ ]:
%sql
-- 4.2 Invoices, Payments & Claims
ALTER TABLE hospital_catalog.silver.invoices
SET TAGS ('contains_financial' = 'true', 'classification' = 'confidential', 'domain' = 'billing');

ALTER TABLE hospital_catalog.silver.invoices ALTER COLUMN patient_id
SET TAGS ('pii' = 'true', 'data_type' = 'pseudonymized_key', 'info_type' = 'patient_mrn');

ALTER TABLE hospital_catalog.silver.invoices ALTER COLUMN gross_amount
SET TAGS ('financial' = 'true', 'data_type' = 'monetary_amount');

ALTER TABLE hospital_catalog.silver.invoices ALTER COLUMN discount_amount
SET TAGS ('financial' = 'true', 'data_type' = 'monetary_amount');

ALTER TABLE hospital_catalog.silver.invoices ALTER COLUMN insurance_covered_amount
SET TAGS ('financial' = 'true', 'data_type' = 'monetary_amount');

ALTER TABLE hospital_catalog.silver.invoices ALTER COLUMN net_amount_due
SET TAGS ('financial' = 'true', 'data_type' = 'monetary_amount');


ALTER TABLE hospital_catalog.silver.payments
SET TAGS ('contains_financial' = 'true', 'classification' = 'confidential', 'domain' = 'billing');

ALTER TABLE hospital_catalog.silver.payments ALTER COLUMN patient_id
SET TAGS ('pii' = 'true', 'data_type' = 'pseudonymized_key', 'info_type' = 'patient_mrn');

ALTER TABLE hospital_catalog.silver.payments ALTER COLUMN amount_paid
SET TAGS ('financial' = 'true', 'data_type' = 'monetary_amount');

ALTER TABLE hospital_catalog.silver.payments ALTER COLUMN payment_method
SET TAGS ('financial' = 'true', 'data_type' = 'monetary_amount', 'info_type' = 'payment_instrument');


ALTER TABLE hospital_catalog.silver.insurance_claims
SET TAGS ('contains_financial' = 'true', 'classification' = 'confidential', 'domain' = 'billing');

ALTER TABLE hospital_catalog.silver.insurance_claims ALTER COLUMN patient_id
SET TAGS ('pii' = 'true', 'data_type' = 'pseudonymized_key', 'info_type' = 'patient_mrn');

ALTER TABLE hospital_catalog.silver.insurance_claims ALTER COLUMN payer_name
SET TAGS ('financial' = 'true', 'data_type' = 'business_entity', 'info_type' = 'insurance_payer');

ALTER TABLE hospital_catalog.silver.insurance_claims ALTER COLUMN claim_amount
SET TAGS ('financial' = 'true', 'data_type' = 'monetary_amount');

In [ ]:
%sql
-- 4.3 Identity Bridge Maps
ALTER TABLE hospital_catalog.silver.patients_map_probabilistic_enterprise
SET TAGS ('contains_pii' = 'true', 'classification' = 'confidential', 'domain' = 'master_data');

ALTER TABLE hospital_catalog.silver.patients_map_probabilistic_enterprise ALTER COLUMN _source_patient_key
SET TAGS ('pii' = 'true', 'data_type' = 'composite_key', 'info_type' = 'composite_branch_patient_id');

ALTER TABLE hospital_catalog.silver.patients_map_probabilistic_enterprise ALTER COLUMN enterprise_patient_id
SET TAGS ('pii' = 'true', 'data_type' = 'pseudonymized_key', 'info_type' = 'enterprise_patient_id');

ALTER TABLE hospital_catalog.silver.patients_map_probabilistic_enterprise ALTER COLUMN source_patient_id
SET TAGS ('pii' = 'true', 'data_type' = 'pseudonymized_key', 'info_type' = 'local_patient_id');


ALTER TABLE hospital_catalog.silver.doctors_map_probabilistic_enterprise
SET TAGS ('contains_pii' = 'true', 'classification' = 'internal', 'domain' = 'master_data');

ALTER TABLE hospital_catalog.silver.doctors_map_probabilistic_enterprise ALTER COLUMN enterprise_doctor_id
SET TAGS ('pii' = 'true', 'data_type' = 'pseudonymized_key', 'info_type' = 'enterprise_doctor_id');

ALTER TABLE hospital_catalog.silver.doctors_map_probabilistic_enterprise ALTER COLUMN source_doctor_id
SET TAGS ('pii' = 'true', 'data_type' = 'pseudonymized_key', 'info_type' = 'local_doctor_id');


## Part 5: Gold Layer — Golden Master Dimensions

In [ ]:
%sql
-- 5.1 gold.dim_patient_enterprise
ALTER TABLE hospital_catalog.gold.dim_patient_enterprise
SET TAGS (
  'contains_pii' = 'true',
  'contains_phi' = 'true',
  'classification' = 'restricted',
  'domain' = 'conformed_dimension',
  'golden_record' = 'true'
);

ALTER TABLE hospital_catalog.gold.dim_patient_enterprise ALTER COLUMN enterprise_patient_id
SET TAGS ('pii' = 'true', 'data_type' = 'pseudonymized_key', 'info_type' = 'enterprise_master_patient_id');

ALTER TABLE hospital_catalog.gold.dim_patient_enterprise ALTER COLUMN first_name
SET TAGS ('pii' = 'true', 'data_type' = 'direct_identifier', 'info_type' = 'first_name');

ALTER TABLE hospital_catalog.gold.dim_patient_enterprise ALTER COLUMN last_name
SET TAGS ('pii' = 'true', 'data_type' = 'direct_identifier', 'info_type' = 'last_name');

ALTER TABLE hospital_catalog.gold.dim_patient_enterprise ALTER COLUMN date_of_birth
SET TAGS ('pii' = 'true', 'phi' = 'true', 'data_type' = 'quasi_identifier', 'info_type' = 'date_of_birth');

ALTER TABLE hospital_catalog.gold.dim_patient_enterprise ALTER COLUMN gender
SET TAGS ('pii' = 'false', 'data_type' = 'quasi_identifier', 'info_type' = 'demographic_gender');

ALTER TABLE hospital_catalog.gold.dim_patient_enterprise ALTER COLUMN contact_no
SET TAGS ('pii' = 'true', 'data_type' = 'direct_identifier', 'info_type' = 'phone_number');

ALTER TABLE hospital_catalog.gold.dim_patient_enterprise ALTER COLUMN address
SET TAGS ('pii' = 'true', 'data_type' = 'direct_identifier', 'info_type' = 'physical_address');

In [ ]:
%sql
-- 5.2 gold.dim_doctor_enterprise
ALTER TABLE hospital_catalog.gold.dim_doctor_enterprise
SET TAGS (
  'contains_pii' = 'true',
  'classification' = 'internal',
  'domain' = 'conformed_dimension',
  'golden_record' = 'true'
);

ALTER TABLE hospital_catalog.gold.dim_doctor_enterprise ALTER COLUMN enterprise_doctor_id
SET TAGS ('pii' = 'true', 'data_type' = 'pseudonymized_key', 'info_type' = 'enterprise_master_doctor_id');

ALTER TABLE hospital_catalog.gold.dim_doctor_enterprise ALTER COLUMN first_name
SET TAGS ('pii' = 'true', 'data_type' = 'direct_identifier', 'info_type' = 'staff_name');

ALTER TABLE hospital_catalog.gold.dim_doctor_enterprise ALTER COLUMN last_name
SET TAGS ('pii' = 'true', 'data_type' = 'direct_identifier', 'info_type' = 'staff_name');

ALTER TABLE hospital_catalog.gold.dim_doctor_enterprise ALTER COLUMN contact_no
SET TAGS ('pii' = 'true', 'data_type' = 'direct_identifier', 'info_type' = 'business_phone_number');


## Part 6: Gold Layer — Fact Tables

In [ ]:
%sql
-- 6.1 fact_appointments
ALTER TABLE hospital_catalog.gold.fact_appointments
SET TAGS (
  'contains_phi' = 'true',
  'classification' = 'restricted',
  'domain' = 'clinical_analytics'
);

ALTER TABLE hospital_catalog.gold.fact_appointments ALTER COLUMN enterprise_patient_id
SET TAGS ('pii' = 'true', 'data_type' = 'pseudonymized_key', 'info_type' = 'enterprise_patient_id');

ALTER TABLE hospital_catalog.gold.fact_appointments ALTER COLUMN enterprise_doctor_id
SET TAGS ('pii' = 'true', 'data_type' = 'pseudonymized_key', 'info_type' = 'enterprise_doctor_id');

ALTER TABLE hospital_catalog.gold.fact_appointments ALTER COLUMN appointment_date
SET TAGS ('phi' = 'true', 'data_type' = 'quasi_identifier', 'info_type' = 'clinical_date');

ALTER TABLE hospital_catalog.gold.fact_appointments ALTER COLUMN reason
SET TAGS ('phi' = 'true', 'data_type' = 'clinical_observation', 'info_type' = 'chief_complaint');

ALTER TABLE hospital_catalog.gold.fact_appointments ALTER COLUMN payment_amount
SET TAGS ('financial' = 'true', 'data_type' = 'monetary_amount');

In [ ]:
%sql
-- 6.2 fact_medical_records
ALTER TABLE hospital_catalog.gold.fact_medical_records
SET TAGS (
  'contains_phi' = 'true',
  'classification' = 'restricted',
  'domain' = 'clinical_analytics'
);

ALTER TABLE hospital_catalog.gold.fact_medical_records ALTER COLUMN enterprise_patient_id
SET TAGS ('pii' = 'true', 'data_type' = 'pseudonymized_key', 'info_type' = 'enterprise_patient_id');

ALTER TABLE hospital_catalog.gold.fact_medical_records ALTER COLUMN enterprise_doctor_id
SET TAGS ('pii' = 'true', 'data_type' = 'pseudonymized_key', 'info_type' = 'enterprise_doctor_id');

ALTER TABLE hospital_catalog.gold.fact_medical_records ALTER COLUMN visit_date
SET TAGS ('phi' = 'true', 'data_type' = 'quasi_identifier', 'info_type' = 'clinical_date');

ALTER TABLE hospital_catalog.gold.fact_medical_records ALTER COLUMN reason
SET TAGS ('phi' = 'true', 'data_type' = 'clinical_observation', 'info_type' = 'chief_complaint');

ALTER TABLE hospital_catalog.gold.fact_medical_records ALTER COLUMN primary_icd10_code
SET TAGS ('phi' = 'true', 'data_type' = 'clinical_observation', 'info_type' = 'icd10_code');

ALTER TABLE hospital_catalog.gold.fact_medical_records ALTER COLUMN primary_diagnosis
SET TAGS ('phi' = 'true', 'data_type' = 'clinical_observation', 'info_type' = 'medical_condition');

In [ ]:
%sql
-- 6.3 fact_admissions
ALTER TABLE hospital_catalog.gold.fact_admissions
SET TAGS (
  'contains_phi' = 'true',
  'classification' = 'restricted',
  'domain' = 'inpatient_analytics'
);

ALTER TABLE hospital_catalog.gold.fact_admissions ALTER COLUMN enterprise_patient_id
SET TAGS ('pii' = 'true', 'data_type' = 'pseudonymized_key', 'info_type' = 'enterprise_patient_id');

ALTER TABLE hospital_catalog.gold.fact_admissions ALTER COLUMN admission_date
SET TAGS ('phi' = 'true', 'data_type' = 'quasi_identifier', 'info_type' = 'admission_date');

ALTER TABLE hospital_catalog.gold.fact_admissions ALTER COLUMN discharge_date
SET TAGS ('phi' = 'true', 'data_type' = 'quasi_identifier', 'info_type' = 'discharge_date');

ALTER TABLE hospital_catalog.gold.fact_admissions ALTER COLUMN ward_name
SET TAGS ('phi' = 'true', 'data_type' = 'location_tracking', 'info_type' = 'hospital_ward');

ALTER TABLE hospital_catalog.gold.fact_admissions ALTER COLUMN room_no
SET TAGS ('phi' = 'true', 'data_type' = 'location_tracking', 'info_type' = 'room_number');

ALTER TABLE hospital_catalog.gold.fact_admissions ALTER COLUMN bed_no
SET TAGS ('phi' = 'true', 'data_type' = 'location_tracking', 'info_type' = 'bed_number');

ALTER TABLE hospital_catalog.gold.fact_admissions ALTER COLUMN discharge_disposition
SET TAGS ('phi' = 'true', 'data_type' = 'clinical_observation', 'info_type' = 'clinical_outcome');

In [ ]:
%sql
-- 6.4 fact_surgeries
ALTER TABLE hospital_catalog.gold.fact_surgeries
SET TAGS (
  'contains_phi' = 'true',
  'classification' = 'restricted',
  'domain' = 'surgical_analytics'
);

ALTER TABLE hospital_catalog.gold.fact_surgeries ALTER COLUMN enterprise_patient_id
SET TAGS ('pii' = 'true', 'data_type' = 'pseudonymized_key', 'info_type' = 'enterprise_patient_id');

ALTER TABLE hospital_catalog.gold.fact_surgeries ALTER COLUMN enterprise_surgeon_id
SET TAGS ('pii' = 'true', 'data_type' = 'pseudonymized_key', 'info_type' = 'enterprise_doctor_id');

ALTER TABLE hospital_catalog.gold.fact_surgeries ALTER COLUMN procedure_name
SET TAGS ('phi' = 'true', 'data_type' = 'clinical_observation', 'info_type' = 'surgical_procedure');

ALTER TABLE hospital_catalog.gold.fact_surgeries ALTER COLUMN procedure_date
SET TAGS ('phi' = 'true', 'data_type' = 'quasi_identifier', 'info_type' = 'clinical_date');

ALTER TABLE hospital_catalog.gold.fact_surgeries ALTER COLUMN notes
SET TAGS ('phi' = 'true', 'data_type' = 'clinical_observation', 'info_type' = 'operative_notes');

In [ ]:
%sql
-- 6.5 fact_lab_results
ALTER TABLE hospital_catalog.gold.fact_lab_results
SET TAGS (
  'contains_phi' = 'true',
  'classification' = 'restricted',
  'domain' = 'diagnostic_analytics'
);

ALTER TABLE hospital_catalog.gold.fact_lab_results ALTER COLUMN enterprise_patient_id
SET TAGS ('pii' = 'true', 'data_type' = 'pseudonymized_key', 'info_type' = 'enterprise_patient_id');

ALTER TABLE hospital_catalog.gold.fact_lab_results ALTER COLUMN enterprise_doctor_id
SET TAGS ('pii' = 'true', 'data_type' = 'pseudonymized_key', 'info_type' = 'enterprise_doctor_id');

ALTER TABLE hospital_catalog.gold.fact_lab_results ALTER COLUMN test_name
SET TAGS ('phi' = 'true', 'data_type' = 'clinical_observation', 'info_type' = 'lab_test_name');

ALTER TABLE hospital_catalog.gold.fact_lab_results ALTER COLUMN parameter_name
SET TAGS ('phi' = 'true', 'data_type' = 'clinical_observation', 'info_type' = 'test_parameter');

ALTER TABLE hospital_catalog.gold.fact_lab_results ALTER COLUMN result_value
SET TAGS ('phi' = 'true', 'data_type' = 'clinical_observation', 'info_type' = 'lab_result_numeric');

ALTER TABLE hospital_catalog.gold.fact_lab_results ALTER COLUMN reference_range
SET TAGS ('phi' = 'true', 'data_type' = 'clinical_observation', 'info_type' = 'reference_range');

ALTER TABLE hospital_catalog.gold.fact_lab_results ALTER COLUMN abnormal_flag
SET TAGS ('phi' = 'true', 'data_type' = 'clinical_observation', 'info_type' = 'abnormal_flag');

ALTER TABLE hospital_catalog.gold.fact_lab_results ALTER COLUMN result_date
SET TAGS ('phi' = 'true', 'data_type' = 'quasi_identifier', 'info_type' = 'clinical_date');

In [ ]:
%sql
-- 6.6 fact_billing
ALTER TABLE hospital_catalog.gold.fact_billing
SET TAGS (
  'contains_financial' = 'true',
  'classification' = 'confidential',
  'domain' = 'financial_analytics'
);

ALTER TABLE hospital_catalog.gold.fact_billing ALTER COLUMN enterprise_patient_id
SET TAGS ('pii' = 'true', 'data_type' = 'pseudonymized_key', 'info_type' = 'enterprise_patient_id');

ALTER TABLE hospital_catalog.gold.fact_billing ALTER COLUMN gross_amount
SET TAGS ('financial' = 'true', 'data_type' = 'monetary_amount');

ALTER TABLE hospital_catalog.gold.fact_billing ALTER COLUMN discount_amount
SET TAGS ('financial' = 'true', 'data_type' = 'monetary_amount');

ALTER TABLE hospital_catalog.gold.fact_billing ALTER COLUMN insurance_covered_amount
SET TAGS ('financial' = 'true', 'data_type' = 'monetary_amount');

ALTER TABLE hospital_catalog.gold.fact_billing ALTER COLUMN net_amount_due
SET TAGS ('financial' = 'true', 'data_type' = 'monetary_amount');

ALTER TABLE hospital_catalog.gold.fact_billing ALTER COLUMN total_amount_paid
SET TAGS ('financial' = 'true', 'data_type' = 'monetary_amount');

ALTER TABLE hospital_catalog.gold.fact_billing ALTER COLUMN outstanding_balance
SET TAGS ('financial' = 'true', 'data_type' = 'monetary_amount');

ALTER TABLE hospital_catalog.gold.fact_billing ALTER COLUMN payer_name
SET TAGS ('financial' = 'true', 'data_type' = 'business_entity', 'info_type' = 'insurance_payer');

ALTER TABLE hospital_catalog.gold.fact_billing ALTER COLUMN total_claim_amount
SET TAGS ('financial' = 'true', 'data_type' = 'monetary_amount');

ALTER TABLE hospital_catalog.gold.fact_billing ALTER COLUMN total_insurance_paid
SET TAGS ('financial' = 'true', 'data_type' = 'monetary_amount');


## Part 7: Verification Queries

In [ ]:
%sql
-- Summary of all table tags in Silver and Gold
SELECT 
    table_catalog,
    table_schema,
    table_name,
    tag_name,
    tag_value
FROM system.information_schema.table_tags
WHERE table_catalog = 'hospital_catalog'
  AND table_schema IN ('silver', 'gold')
ORDER BY table_schema, table_name, tag_name;

In [ ]:
%sql
-- Summary of all column tags in Silver and Gold
SELECT 
    table_schema,
    table_name,
    column_name,
    tag_name,
    tag_value
FROM system.information_schema.column_tags
WHERE table_catalog = 'hospital_catalog'
  AND table_schema IN ('silver', 'gold')
ORDER BY table_schema, table_name, column_name;